In [0]:
%pip install swift-parser-py

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26304 sha256=7ec929f4cb1c2853f6a61b12cd97c7aeeeaaae891548c411fd2c49cab91431cf
  Stored in directory: /home/spark-d65a07dd-f2c3-44b9-b4ab-4b/.cache/pip/wheels/12/98/52/2bffe242a9a487f00886e43b8ed8dac46456702e11a0d6abef
Successfully built typing
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook or Python script
from pyspark.sql import SparkSession
import xml.etree.ElementTree as ET
import json

# -------------------------------
# 1. Spark session (Databricks provides one automatically in notebooks)
# -------------------------------
spark = SparkSession.builder.getOrCreate()

# -------------------------------
# 2. Example SWIFT MX (ISO 20022) message
# -------------------------------
mx_message = """
<Document xmlns="urn:iso:std:iso:20022:tech:xsd:pacs.008.001.07">
    <FIToFICstmrCdtTrf>
        <GrpHdr>
            <MsgId>ABC123456</MsgId>
            <CreDtTm>2026-03-04T10:15:00</CreDtTm>
        </GrpHdr>
        <CdtTrfTxInf>
            <PmtId>
                <EndToEndId>E2E-REF-001</EndToEndId>
            </PmtId>
            <IntrBkSttlmAmt Ccy="USD">1500.00</IntrBkSttlmAmt>
            <Cdtr>
                <Nm>John Doe</Nm>
            </Cdtr>
        </CdtTrfTxInf>
    </FIToFICstmrCdtTrf>
</Document>
"""

# -------------------------------
# 3. Parse MX message using XML parser
# -------------------------------
# Parse the XML
root = ET.fromstring(mx_message)

# Define namespace
ns = {'ns': 'urn:iso:std:iso:20022:tech:xsd:pacs.008.001.07'}

# -------------------------------
# 4. Extract relevant fields from parsed XML
# -------------------------------
msg_id_elem = root.find('.//ns:MsgId', ns)
end_to_end_id_elem = root.find('.//ns:EndToEndId', ns)
amount_elem = root.find('.//ns:IntrBkSttlmAmt', ns)
creditor_name_elem = root.find('.//ns:Cdtr/ns:Nm', ns)

msg_id = msg_id_elem.text if msg_id_elem is not None else None
end_to_end_id = end_to_end_id_elem.text if end_to_end_id_elem is not None else None
amount = float(amount_elem.text) if amount_elem is not None else 0.0
currency = amount_elem.get('Ccy') if amount_elem is not None else None
creditor_name = creditor_name_elem.text if creditor_name_elem is not None else None

parsed_data = [(msg_id, end_to_end_id, amount, currency, creditor_name)]

# -------------------------------
# 5. Create Spark DataFrame from parsed data
# -------------------------------
columns = ["MsgId", "EndToEndId", "Amount", "Currency", "CreditorName"]
parsed_df = spark.createDataFrame(parsed_data, columns)

# -------------------------------
# 6. Show parsed results
# -------------------------------
display(parsed_df)

MsgId,EndToEndId,Amount,Currency,CreditorName
ABC123456,E2E-REF-001,1500.0,USD,John Doe
